In [73]:
from sipyco.pc_rpc import Client
from driver_topticadlc import TopticaDLCPro

remote = Client("137.222.69.28", 3272, "TopticaDLCPro", timeout=1)

In [74]:
print(remote.get_local_host())
print(remote.get_selected_target())
remote._Client__valid_methods


137.222.69.28
TopticaDLCPro


{'laser2_nlo_power_optimization_stage5_input_value_calibrated',
 'laser1_nlo_shg_cavity_tc_current_set_min',
 'laser1_scope_channelx_scope_timescale_set',
 'laser2_uv_scope_channel2_name',
 'laser1_dl_motor_position_set',
 'tc1_serial_number',
 'laser2_uv_pump_temperature_control_t5_temp_set',
 'laser1_nlo_shg_lock_window_level_hysteresis',
 'laser2_dl_eom_external_input_signal_set',
 'laser3_nlo_opo_cavity_tc_temp_set_min_set',
 'laser3_amp_factory_settings_modified',
 'laser4_hf_cavity_tc1_limits_timed_out',
 'laser3_dl_tc_current_set',
 'laser3_amp_seed_limits_power_max_shutdown_delay_set',
 'laser1_amp_ontime_txt',
 'laser3_nlo_fhg_lock_pid2_gain_i',
 'ampcc1_serial_number',
 'laser1_nlo_fhg_factory_settings_tc_timeout',
 'laser4_nlo_fhg_scope_data',
 'laser3_dl_lock_pid2_sign',
 'laser1_nlo_fhg_tc_c_loop_i_gain_set',
 'laser4_dpss_temperature_control_t4_temp_set_min_set',
 'laser4_dl_lock_window_level_low_set',
 'laser1_dl_lock_candidate_filter_top',
 'laser3_nlo_opo_factory_setti

Connect and Read the data from DLC Pro 

In [ ]:
# For xy scans
from toptica.lasersdk.utils.dlcpro import (
    extract_float_arrays,
    extract_lock_points,
    extract_lock_state,
)
import matplotlib.pyplot as plt

with TopticaDLCPro(ip="192.168.0.4") as remote:
    laser = remote.laser2
    if laser.scope.channel1.signal.get():
        scope_data = extract_float_arrays("xyY", laser.scope.data.get())

        raw_lock_candidates = laser.dl.lock.candidates.get()
        lock_candidates = extract_lock_points("clt", raw_lock_candidates)
        lock_state = extract_lock_state(raw_lock_candidates)

        # Get background data
        background_data = extract_float_arrays(
            "xy", laser.dl.lock.background_trace.get()
        )

        # Data now available in scope_data, lock_candidates
        # in lock_candidates['c'] are all the potential points, and 'l' the selected point
        # in scope_data you have 'y' the lock signal, and 'Y' the derivative

    else:
        print("No scope data available for laser 2.")
print(lock_candidates)

#plot x and y data
plt.figure(figsize=(10, 6))
plt.plot(scope_data["x"], scope_data["y"], label="Lock Signal")
#plt.plot(scope_data["x"], scope_data["Y"], label="Derivative")
plt.scatter(
    lock_candidates["c"]["x"],
    lock_candidates["c"]["y"],
    color="red",
    label="Lock Candidates",
)
# plt.scatter(
#     lock_candidates["l"]["x"],
#     lock_candidates["l"]["y"],
#     color="green",
#     label="Selected Lock Point",
# )
plt.title("Lock Signal and Derivative with Lock Candidates")
plt.xlabel("Frequency (MHz)")
plt.ylabel("Signal")
plt.legend()
plt.grid()
plt.show()

AttributeError: 'ScopeT' object has no attribute 'resolution'

In [ ]:
# for f scans
from toptica.lasersdk.utils.dlcpro import (
    extract_float_arrays,
    extract_lock_points,
    extract_lock_state,
)
from driver_topticadlc import TopticaDLCPro

with TopticaDLCPro(ip="192.168.0.4") as remote:
    laser = remote.laser2
    if laser.scope.channel1.signal.get():
        scope_data = extract_float_arrays("t", laser.scope.data.get())

    #     raw_lock_candidates = laser.dl.lock.candidates.get()
    #     lock_candidates = extract_lock_points("clt", raw_lock_candidates)
    #     lock_state = extract_lock_state(raw_lock_candidates)

    #     # Get background data
    #     background_data = extract_float_arrays(
    #         "f", laser.dl.lock.background_trace.get()
    #     )

    #     # Data now available in scope_data, lock_candidates
    #     # in lock_candidates['c'] are all the potential points, and 'l' the selected point
    #     # in scope_data you have 'y' the lock signal, and 'Y' the derivative

    # else:
    #     print("No scope data available for laser 1.")
print(scope_data)

{}


Plot the lock-points 

In [ ]:
# Unpack lockpoint data
candidate_x = lock_candidates["c"]["x"]
candidate_y = lock_candidates["c"]["y"]
selected_x = lock_candidates["l"]["x"]
selected_y = lock_candidates["l"]["y"]

# Unpack scope signal (from photodiode or saturated absorption etc.)
scope_x = scope_data["x"]
scope_y = scope_data["y"]

# Plot scope trace
plt.plot(scope_x, scope_y, label="Scope Signal", color="blue")

# Plot all lock candidates
plt.scatter(
    candidate_x, candidate_y, label="Lock Candidates", color="orange", marker="x"
)

# Plot selected lock point(s)
plt.scatter(
    selected_x,
    selected_y,
    label="Selected Lockpoint",
    color="red",
    marker="o",
    s=80,
    edgecolors="black",
)

# Final plot formatting
plt.xlabel("Frequency (or Scan Position)")
plt.ylabel("Signal (Arb. Units)")
plt.title("TOPTICA Lock Signal with Lock Candidates_Starting Point")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

KeyError: 'x'

Save the data to an CSV for further analysis 

In [ ]:
import csv
import json
from datetime import datetime
import os

# Get timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Define filenames
csv_filename = f"selected_lockpoints_{timestamp}_77_0.csv"
json_filename = f"selected_lockpoints_{timestamp}_77_0.json"
full_json_filename = f"all_lockpoints_{timestamp}_77_0.json"

# Optional: choose output directory
output_dir = "lockpoint_logs"
os.makedirs(output_dir, exist_ok=True)

# Save selected lockpoints to CSV
with open(os.path.join(output_dir, csv_filename), mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["x", "y"])  # header
    for x_val, y_val in zip(selected_x, selected_y):
        writer.writerow([x_val, y_val])

# Save selected lockpoints to JSON
selected_points = {"x": selected_x, "y": selected_y}
with open(os.path.join(output_dir, json_filename), "w") as f:
    json.dump(selected_points, f, indent=2)

# Save all lock candidates (optional)
with open(os.path.join(output_dir, full_json_filename), "w") as f:
    json.dump(lock_candidates, f, indent=2)

print(f"Saved selected lockpoints to {csv_filename} and {json_filename}")

Saved selected lockpoints to selected_lockpoints_20250721_133030.csv and selected_lockpoints_20250721_133030.json
